# XLM-R base — MGT Detection Fine-tuning (Baseline)
**COS760 Group 57 | XLM-RoBERTa-base classifier for isiZulu / English / Combined / Cross-lingual**

This is the **XLM-R baseline** — run this alongside AfroXLMR_Colab.ipynb to compare performance.
The only difference from AfroXLMR is the model (`xlm-roberta-base` vs `Davlan/afro-xlmr-base`).

---

### Before running:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU → Save
2. **Run cells top to bottom**, one at a time
3. **To switch experiments**, change `EXPERIMENT` in Cell 4 and re-run from Cell 5 onward

| `EXPERIMENT` | Dataset | Purpose |
|---|---|---|
| `'zul'` | zul_train/val/test.csv | isiZulu-only (RQ1) |
| `'eng'` | eng_train/val/test.csv | English-only baseline |
| `'combined'` | combined_train/val/test.csv | Both languages |
| `'cross'` | Train: eng / Test: zul | Cross-lingual transfer (RQ2) |

## Cell 1 — Install packages

In [2]:
!pip install transformers datasets evaluate scikit-learn accelerate scipy -q
print('Packages installed')

Packages installed


## Cell 2 — Verify GPU
Should print `Tesla T4 | 15.8 GB`. If it shows CPU, go to Runtime → Change runtime type → T4 GPU.

In [3]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  |  {vram:.1f} GB VRAM')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

GPU: Tesla T4  |  15.6 GB VRAM


## Cell 3 — Upload CSV files
Select all 9 CSV files at once (hold Ctrl/Cmd to multi-select).
If you already uploaded them for AfroXLMR in the same session, skip this cell — the files are still at `/content/data`.

In [4]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)

if os.listdir('/content/data'):
    print('Files already present:', os.listdir('/content/data'))
    print('Skipping upload. If you want to re-upload, clear /content/data first.')
else:
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, f'/content/data/{fname}')
    print('Files uploaded:', os.listdir('/content/data'))

DATA_DIR = '/content/data'

Files already present: ['zul_val.csv', 'combined_test.csv', 'combined_val.csv', 'eng_test.csv', 'zul_test.csv', 'eng_val.csv', 'zul_train.csv', 'eng_train.csv', 'combined_train.csv']
Skipping upload. If you want to re-upload, clear /content/data first.


## Cell 4 — Experiment configuration
**Change `EXPERIMENT` here to switch between runs.** Re-run from Cell 5 onward after changing it.

In [5]:
# CHANGE THIS to switch experiments
EXPERIMENT = 'zul'   # 'zul' | 'eng' | 'combined' | 'cross'

MODEL_NAME  = 'xlm-roberta-base'   # <-- only line different from AfroXLMR notebook
DATA_DIR    = '/content/data'
OUTPUT_DIR  = f'/content/outputs/xlmr_base_{EXPERIMENT}'
BATCH_SIZE  = 16
MAX_LEN     = 256
EPOCHS      = 5
LR          = 2e-5
SEED        = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Experiment : {EXPERIMENT}')
print(f'Model      : {MODEL_NAME}')
print(f'Output     : {OUTPUT_DIR}')

Experiment : zul
Model      : xlm-roberta-base
Output     : /content/outputs/xlmr_base_zul


## Cell 5 — Load and inspect data

In [6]:
import pandas as pd
from datasets import Dataset

def load_csv(name):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing: {path}')
    return pd.read_csv(path)

if EXPERIMENT == 'cross':
    train_df = load_csv('eng_train.csv')
    val_df   = load_csv('eng_val.csv')
    test_df  = load_csv('zul_test.csv')
else:
    train_df = load_csv(f'{EXPERIMENT}_train.csv')
    val_df   = load_csv(f'{EXPERIMENT}_val.csv')
    test_df  = load_csv(f'{EXPERIMENT}_test.csv')

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    h = (df['label'] == 0).sum()
    m = (df['label'] == 1).sum()
    print(f'{name:5s}: {len(df)} samples  (human={h}, machine={m})')

Train: 946 samples  (human=473, machine=473)
Val  : 203 samples  (human=101, machine=102)
Test : 203 samples  (human=102, machine=101)


## Cell 6 — Tokenise data
Downloads XLM-R base on first run (~1.1 GB). Subsequent runs use the cached copy.

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(df):
    ds = Dataset.from_dict({
        'text':  df['text'].astype(str).tolist(),
        'label': df['label'].astype(int).tolist(),
    })
    def tok(batch):
        return tokenizer(batch['text'], padding='max_length',
                         truncation=True, max_length=MAX_LEN)
    ds = ds.map(tok, batched=True)
    ds = ds.remove_columns(['text'])
    ds = ds.rename_column('label', 'labels')
    ds.set_format('torch')
    return ds

train_ds = make_hf_dataset(train_df)
val_ds   = make_hf_dataset(val_df)
test_ds  = make_hf_dataset(test_df)
print('Tokenisation complete')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/946 [00:00<?, ? examples/s]

Map:   0%|          | 0/203 [00:00<?, ? examples/s]

Map:   0%|          | 0/203 [00:00<?, ? examples/s]

Tokenisation complete
Train: 946 | Val: 203 | Test: 203


## Cell 7 — Load model

In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
)
print(f'Model loaded: {MODEL_NAME}')
total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters : {total_params/1e6:.1f}M')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: xlm-roberta-base
Parameters : 278.0M


## Cell 8 — Define metrics

In [9]:
import numpy as np
import scipy.special
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = scipy.special.softmax(logits, axis=-1)[:, 1]
    return {
        'accuracy':  round(accuracy_score(labels, preds), 4),
        'f1':        round(f1_score(labels, preds, average='macro'), 4),
        'precision': round(precision_score(labels, preds, average='macro', zero_division=0), 4),
        'recall':    round(recall_score(labels, preds, average='macro', zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(labels, probs), 4),
    }

print('Metrics function defined')

Metrics function defined


## Cell 9 — Train
Training takes ~20 min per experiment on a T4 GPU.
You will see `eval_f1` printed after each epoch — it should increase. Early stopping triggers if F1 does not improve for 2 consecutive epochs.

In [10]:
import torch
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

torch.manual_seed(SEED)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

best_f1 = trainer.state.best_metric
print(f'\nBest val F1: {best_f1:.4f}', 'Good' if best_f1 > 0.55 else 'Near random — check training logs')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
1,0.683700,0.603637,0.803000,0.801200,0.815100,0.803400,0.928000
2,0.490034,0.370318,0.842400,0.838100,0.880600,0.841600,0.929900
3,0.229096,0.392036,0.886700,0.885100,0.908000,0.886100,0.984600
4,0.160016,0.508080,0.891600,0.890200,0.911300,0.891100,0.985000
5,0.064936,0.333086,0.926100,0.925600,0.935900,0.925700,0.986100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Best val F1: 0.9256 Good


## Cell 10 — Evaluate on test set
Reports accuracy, F1, precision, recall, ROC-AUC, confusion matrix, and majority class baseline.

In [11]:
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

preds_out = trainer.predict(test_ds)
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids
probs  = scipy.special.softmax(preds_out.predictions, axis=-1)[:, 1]

run_name = f'xlmr_{EXPERIMENT}'
print('='*55)
print(f'  TEST RESULTS  —  {run_name}')
print('='*55)
print(classification_report(labels, preds, target_names=['Human', 'Machine']))
print(f'ROC-AUC : {roc_auc_score(labels, probs):.4f}')
print('Confusion Matrix:')
print(confusion_matrix(labels, preds))
print('='*55)

majority  = Counter(labels.tolist()).most_common(1)[0][0]
maj_preds = [majority] * len(labels)
print(f'\nMajority baseline F1: {f1_score(labels, maj_preds, average="macro"):.4f}  (your model must beat this)')

  TEST RESULTS  —  xlmr_zul
              precision    recall  f1-score   support

       Human       1.00      0.88      0.94       102
     Machine       0.89      1.00      0.94       101

    accuracy                           0.94       203
   macro avg       0.95      0.94      0.94       203
weighted avg       0.95      0.94      0.94       203

ROC-AUC : 0.9976
Confusion Matrix:
[[ 90  12]
 [  0 101]]

Majority baseline F1: 0.3344  (your model must beat this)


## Cell 11 — Save results and download
Saves the test metrics CSV and best model checkpoint. Downloads the CSV to your laptop.

In [12]:
from google.colab import files as colab_files

results_df = pd.DataFrame([{
    'run':       run_name,
    'model':     'xlm-roberta-base',
    'lang':      EXPERIMENT,
    'accuracy':  accuracy_score(labels, preds),
    'f1_macro':  f1_score(labels, preds, average='macro'),
    'precision': precision_score(labels, preds, average='macro'),
    'recall':    recall_score(labels, preds, average='macro'),
    'roc_auc':   roc_auc_score(labels, probs),
}])

results_path = os.path.join(OUTPUT_DIR, f'test_results_{EXPERIMENT}.csv')
results_df.to_csv(results_path, index=False)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

trainer.save_model(os.path.join(OUTPUT_DIR, 'best_model'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'best_model'))

colab_files.download(results_path)

     run            model lang  accuracy  f1_macro  precision  recall  roc_auc
xlmr_zul xlm-roberta-base  zul    0.9409    0.9407     0.9469  0.9412   0.9976


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Cell 12 — (Optional) Run all 4 experiments sequentially
**If the session disconnects you will lose all progress. Running Cells 4–11 one experiment at a time is safer.**

Requires Cells 1–8 to have been run first.

In [ ]:
# !! Only run this if you want to do all 4 experiments automatically !!

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from collections import Counter

all_results = []

for exp in ['zul', 'eng', 'combined', 'cross']:
    print(f'\n{"="*55}\n  Experiment: {exp}\n{"="*55}')

    if exp == 'cross':
        tr = load_csv('eng_train.csv')
        va = load_csv('eng_val.csv')
        te = load_csv('zul_test.csv')
    else:
        tr = load_csv(f'{exp}_train.csv')
        va = load_csv(f'{exp}_val.csv')
        te = load_csv(f'{exp}_test.csv')

    tr_ds = make_hf_dataset(tr)
    va_ds = make_hf_dataset(va)
    te_ds = make_hf_dataset(te)

    model_run = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
    )

    out = f'/content/outputs/xlmr_base_{exp}'
    os.makedirs(out, exist_ok=True)

    args_run = TrainingArguments(
        output_dir=out,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to='none',
    )

    t = Trainer(
        model=model_run, args=args_run,
        train_dataset=tr_ds, eval_dataset=va_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    t.train()

    best_f1 = t.state.best_metric
    print(f'  Best val F1: {best_f1:.4f}', 'Good' if best_f1 > 0.55 else 'Near random')

    p_out  = t.predict(te_ds)
    preds  = np.argmax(p_out.predictions, axis=-1)
    labels = p_out.label_ids
    probs  = scipy.special.softmax(p_out.predictions, axis=-1)[:, 1]

    majority  = Counter(labels.tolist()).most_common(1)[0][0]
    maj_preds = [majority] * len(labels)

    result = {
        'run':         f'xlmr_base_{exp}',
        'model':       'xlm-roberta-base',
        'lang':        exp,
        'accuracy':    accuracy_score(labels, preds),
        'f1_macro':    f1_score(labels, preds, average='macro'),
        'precision':   precision_score(labels, preds, average='macro'),
        'recall':      recall_score(labels, preds, average='macro'),
        'roc_auc':     roc_auc_score(labels, probs),
        'majority_f1': f1_score(labels, maj_preds, average='macro'),
    }
    all_results.append(result)

    pd.DataFrame([result]).to_csv(f'{out}/test_results_{exp}.csv', index=False)
    t.save_model(f'{out}/best_model')
    tokenizer.save_pretrained(f'{out}/best_model')
    print(f'  Test F1: {result["f1_macro"]:.4f}  |  Majority baseline: {result["majority_f1"]:.4f}')

    del model_run, t
    torch.cuda.empty_cache()

summary = pd.DataFrame(all_results)
print('\n' + '='*70)
print(summary[['run','f1_macro','roc_auc','accuracy','majority_f1']].to_string(
    index=False, float_format=lambda x: f'{x:.4f}'
))

summary.to_csv('/content/outputs/all_base_xlmr_results.csv', index=False)
colab_files.download('/content/outputs/all_base_xlmr_results.csv')


  Experiment: zul


Map:   0%|          | 0/946 [00:00<?, ? examples/s]

Map:   0%|          | 0/203 [00:00<?, ? examples/s]

Map:   0%|          | 0/203 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Map:   0%|          | 0/209 [00:00<?, ? examples/s]

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
